## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 122 samples and 11995 columns.
First few rows of the data:


,Time to Recurrence,Progression,Days_to_Death or Last Contact,Dead_or_alive,Study_ID,Race,Male_1,stage,smoking_Pack_Years,Age_range,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,656,1,1141,1,20,White,1,IA,50.0,70-79,...,7.286328,4.373904,6.597366,7.722683,5.046826,6.580865,7.924363,7.167449,5.838908,7.102727
1,4661,0,4661,0,32,White,2,IA,5.0,60-69,...,7.065119,4.213735,6.424642,7.146611,5.949407,6.675288,7.809343,7.278092,5.831526,7.081395
2,2319,0,2319,0,50,White,1,1,7.5,70-79,...,9.408223,6.913720,9.183168,9.787556,8.098887,9.210414,10.177154,9.752128,8.031843,9.767943
3,4610,0,4610,0,62,White,1,1,45.0,60-69,...,7.675236,5.578374,7.355767,8.249087,6.519670,7.328719,8.565213,8.322208,6.614988,8.005188
4,3030,0,3030,0,113,White,2,1,35.0,70-79,...,7.829981,5.514712,7.691057,8.239129,6.501239,7.554302,8.892646,8.319649,6.650228,8.282383


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
